In [1]:
import pandas as pd
import numpy as np
from collections import Counter

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is decremented at the begining of the year, and re-calcuated at the begining of the year.

##### Load and setup demand

In [2]:
# Load the CSV file
file_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(file_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [3]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
s_start = pd.read_excel(file_path, sheet_name='S_start')

##### Initialize stuff

In [4]:
#tender length
delta = 3
vaccine_consumption_percent = 1
years = 10

# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store unvaccinated children


##### Initialize antigens/vaccines/prodicers

In [5]:
A = ["Measles", "Mumps", "Rubella"]
V = ["M", "MR", "MMR"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"]
}

P = ["Biological_E", 
    "GSK","PT_Bio", 
    "Serum_Institute"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}

#translate vaccine - antigen, to antigen - vaccine
V_a = {}

for a in A:
    vector_a = []
    for v in A_v.keys():
        if a in A_v[v]:
            vector_a.append(v)
    V_a[a] = vector_a

V_a



{'Measles': ['M', 'MR', 'MMR'], 'Mumps': ['MMR'], 'Rubella': ['MR', 'MMR']}

## TESTING - Measles Containing Vaccines Only

In [6]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
f_start_MCV = f_start[f_start['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
s_start_MCV = s_start[s_start['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = ratio_DF[ratio_DF['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]


In [7]:
demand_80_MCV

,antigen,1,2,3,4,5,6,7,8,9,10
10,Measles,420097000,358616900,394831600,346093200,377569100,407486200,271512200,386591100,290515900,326808200
12,Mumps,26052200,26626400,25785400,25268100,23536200,16205400,18650400,8909600,10718500,12977700
20,Rubella,357331300,254011400,304621400,276893800,291347300,386542700,259074600,385691400,289592400,325856600


In [8]:
inventory_DF_MCV

,Vaccine,Amount
4,M,257581400.0
5,MR,837500100.0
6,MMR,78464000.0


##### Logic to translate vaccine totals to antigen coverage for later math

In [9]:
# Initialize the total coverage dictionary
total_coverage = {antigen: 0 for antigen in V_a.keys()}

# Iterate across each row in the DataFrame
for index, row in inventory_DF_MCV.iterrows():
    vaccine = row['Vaccine']
    amount = row['Amount']
    
    # For each antigen covered by the vaccine, add the amount to the coverage
    for antigen in V_a.keys():
        if vaccine in V_a[antigen]:
            total_coverage[antigen] += amount

# Convert the total coverage dictionary to a DataFrame
total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])

# Output the total coverage for each antigen
total_coverage_df


,antigen,Total_Coverage
0,Measles,1.173546e+09
1,Mumps,7.846400e+07
2,Rubella,9.159641e+08


In [10]:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# Create a list of antigens sorted by their occurrence - sorted from least covered to most covered by vaccine
least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)


In [11]:
demand_80_MCV.iloc[:, [0, 2]]

,antigen,2
10,Measles,358616900
12,Mumps,26626400
20,Rubella,254011400


In [12]:
total_coverage_df

,antigen,Total_Coverage
0,Measles,1.173546e+09
1,Mumps,7.846400e+07
2,Rubella,9.159641e+08


In [13]:
demand = demand_80_MCV.iloc[:, [0, 2]]
demand

,antigen,2
10,Measles,358616900
12,Mumps,26626400
20,Rubella,254011400


In [16]:
calculate_ratios_DF = pd.merge(demand_80_MCV.iloc[:, [0, 2]], total_coverage_df)
calculate_ratios_DF['Ratio'] = calculate_ratios_DF['Total_Coverage']/calculate_ratios_DF[2]
calculate_ratios_DF['Ratio']

0    3.272421
1    2.946850
2    3.605996
Name: Ratio, dtype: float64

In [17]:
calculate_ratios_DF

,antigen,2,Total_Coverage,Ratio
0,Measles,358616900,1.173546e+09,3.272421
1,Mumps,26626400,7.846400e+07,2.946850
2,Rubella,254011400,9.159641e+08,3.605996


In [20]:
for year in range(1, 1 + 1):  # Iterate through each year
    print(f"Year: {year}")

    ## NEW ############################
    #reset ratio DF
    ratio_DF_MCV.loc[:, 1] = np.zeros(len(A))

    # Initialize the total coverage dictionary
    total_coverage = {antigen: 0 for antigen in V_a.keys()}

    # Iterate across each row in the DataFrame
    for index, row in inventory_DF_MCV.iterrows():
        vaccine = row['Vaccine']
        amount = row['Amount']
        
        # For each antigen covered by the vaccine, add the amount to the coverage
        for antigen in V_a.keys():
            if vaccine in V_a[antigen]:
                total_coverage[antigen] += amount

    # Convert the total coverage dictionary to a DataFrame
    total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])

    # Output the total coverage for each antigen
    total_coverage_df

    #Calculate ratio of supply and demand. If demand > supply, schedule a new tender, then move on to update supply and demand
    #update tender schedule, next year +2 (2 total years).
    #AT END OF IF STATEMENT - update inventory
    calculate_ratios_DF = pd.merge(demand_80_MCV.iloc[:, [0, year]], total_coverage_df)
    calculate_ratios_DF['Ratio'] = calculate_ratios_DF['Total_Coverage']/calculate_ratios_DF[1]
    print(f"ratio DF: {calculate_ratios_DF['Ratio']}")


    ###############################################################

    for antigen in least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        print(f"Antigen: {antigen}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen

            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year] >0:
                print (vaccine)
                print(antigens)
                remaining_inventory = {} #is this the correct place?
                print("---------------------------------------------")
                # print(f"Vaccine: {vaccine} covers {antigen}")
                
                # compute inventory of vaccine
                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine].iloc[0, 1]
                print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year]
                print(f"Demand for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                if difference > 0:
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else:
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                print(f"inventory remaining: {remaining_inventory}")
                # Safely update the inventory value for the specific vaccine
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, inventory_DF_MCV.columns[year]] = remaining_inventory[vaccine]

                #decrement all antigens:
                for antigen in antigens:
                    demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen, demand_80_MCV.columns[year]] = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year] - decrement
            # else: - THIS LOGIC IS INCORRECT, because of the antigen in antigens AND!
            #     print(f"Demand is: {demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year]}")
            #     raise ValueError(f"Negative demand detected for {antigen}")

    #If demand remaining > 0, transfer demand to next year
    
    #if demand remaining > 0, add to count of unvax children


                

Year: 1
ratio DF: 0    2.793511
1    3.011799
2    2.563347
Name: Ratio, dtype: float64
###############################################################
Antigen: Mumps
MMR
['Measles', 'Mumps', 'Rubella']
---------------------------------------------
Inventory for MMR for year 1:  78464000.0
Demand for Mumps for year 1:  26052200
inventory remaining: {'MMR': 52411800.0}
###############################################################
Antigen: Rubella
MR
['Measles', 'Rubella']
---------------------------------------------
Inventory for MR for year 1:  837500100.0
Demand for Rubella for year 1:  331279100
inventory remaining: {'MR': 506221000.0}
###############################################################
Antigen: Measles
M
['Measles']
---------------------------------------------
Inventory for M for year 1:  257581400.0
Demand for Measles for year 1:  62765700
inventory remaining: {'M': 194815700.0}


In [ ]:
demand_80_MCV

In [ ]:
demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == 'Measles', demand_80_MCV.columns[2]]